In [1]:
# '취업'에 대해 학습 시키켜록한다
# 네이버 지식인에서 취업관 관련된 질문들을 가져와서 학습시키기 전 단계까지 처리하려고 한다

# 네이버 지식인에서 취업관 관련된 질문 200개를 추출해서 갸져오는 작업
import requests
from bs4 import BeautifulSoup
import pandas as pd

# 매개변수
# 변수명 : 타입 = 기본값
# def함수() -> 리턴타입:
def get_kin_simmary(keyword : str,page : int = 1)->pd.DataFrame:
	# URL 지정
	url = f"https://kin.naver.com/search/list.naver"
	# url뒤에 ?변수 = 값 형태로 만들기 위한 딕셔너리
	# 이 딕셔너리 이름은 반드시 parmas? X
	params = {
		'query' : keyword,
		'page' : page
	}
	# 왼쪽 params는 키워드 인자이가 때문반드시 params
	response = requests.get(url, params=params)

	try:
		# 상태 코드가 200이면 아무것도 없음. 200이 아니면 예외 발생
		response.raise_for_status()
		# 확인용
		# print(response.text)
		# Beautifulsoup 객체 생성

		# html 코드에서 원하는 요소를 탐색하기 위해 bs를 이용
		soup = BeautifulSoup(response.text, 'lxml')# lxml : html코드, lxml-xml

		#._searchListTitleAnchor들을 선택
		# select, semect_one : 선택자 기반으로 요소 선택(보통 html에서)
		# find_all, find : 태그 기반으로 요소 선택(보통 xml에서)
		link_elements = soup.select('._searchListTitleAnchor') # 최대 10개

		# 결과를 담을 리스트
		link_list, title_list = [], []

		# 반복문으로 선택한 요소를 활용
		for link_element in link_elements:
			# 확인용
			# print(link_element)
			title = link_element.text # 요소 안에 있는태그들은 무시한 순수 글자들
			# 요소.get('속성') : 속성 값을 가져옴
			link = link_element.get('href')

			# 확인용
			# print(title)
			# print(link)

			# 추출한 데이터를 각 리스트에 추가
			link_list.append(link)
			title_list.append(title)

			# 확인용
			# print(link_list)
			# df : 변수이름(dataFrame)
			# 딕셔너리 : {'변수명' : 값, '변수명' : 값}
			# 데이터 처리를 위한 데이터 프레임 생성
			df = pd.DataFrame({
				'title' : title_list,
				'link' : link_list
			})

			# 확인용
			# print(df)
		return df
	except Exception as e:
		print(f'예외발생 : {e}')
		return pd.DataFrame({
			'title' : [],
				'link' : []
		})
	

In [ ]:
def get_content(url):
	headers = {
		'User-Agent' : 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/145.0.0.0 Safari/537.36',
	}
	response = requests.get(url, headers=headers)
	try:
		response.raise_for_status()

	except Exception as e:
		print(f"예외 발생 {e}")
		return None

	soup = BeautifulSoup(response.text, 'lxml')
	content = soup.select_one('.questionDetail')

	return content.text

# print(get_kin_simmary(keyword, 1))
import pandas as pd
import time
max_page = 20
keyword = '취업'

# 모든 결과를 담을 데이터프레임
total_df = pd.DataFrame({
	'title' : [], 
	'link' : [],
	'content' : []
})
# 200개 가져올 때까지 반복 => 기본 10*20페이지
for page in range(1, max_page + 1):
	"""지식인 질문들을 질문 제목(title)과 내용(content)으로 구성된 데이터프레임으로 반환하는 """
	# 키워드, 페이지를 이용하여 검색된 결과에서 제목과 내용을 추출
	df = get_kin_simmary(keyword, page)

	contents = []
	# df에서 지직인 링크 목록을 가져와서 반복문으로 실행
	for link in df['link']:
		# { 'title' : "제목", 'content' : '질문 내용' }
		content = get_content(link) if link else None
		contents.append(content.strip())

	df['content'] = contents

	total_df = total_df.dropna()

	# 확인용
	# print(df)
	# 질문을 추출 => 질문이 어디에 있는지

	# 각 페이지 결과를 전체에 이어붙임
	total_df = pd.concat([total_df, df], axis=0)
	total_df = total_df.drop(columns=['link'])
	total_df.to_csv(f'{keyword}content.csv', encoding="utf-8-sig", index=False)	

	# 지연
	time.sleep(0.5)

In [ ]:
# index 리셋
total_df = total_df.reset_index(drop=True)
# print(total_df)

                         title  \
0            실럽급여 받는 중 취업 햇을 시   
1       국민취업지원제도는 어떻게 진행을 하나요?   
2        음주운전으로 미국취업비자 발급이...    
3           E7비자 취업만하면 받을수 있나요   
4                  공기업(공공기관)취업   
5      실업급여 받는 중에 취업하면 어떻게...    
6                     성범죄자 취업시   
7   실업급여 수급 중 조기 취업 시 불이익은...    
8          실업급여 받던중 취업예정일이...    
9                     간호조무사 취업   
10       운전면허 취소 보육교사 취업 가능한가요   
11              실업급여 조기취업수당 질문   
12                토목기사 취득 및 취업   
13        연류가 됫는데 경비에 취업이 되나요?   
14    소방안전관리자2급 자격증으로 취업가능...    
15           집유 끝나고 공공기관 취업 질문   
16   대학교 3학년에 취업하면 자퇴밖에 답이...    
17              공기업 회계팀 취업시 스펙   
18               조기취업수당 해당되나요?   
19                 실업급여 끝나고 취업   

                                                 link  
0   https://kin.naver.com/qna/detail.naver?d1id=6&...  
1   https://kin.naver.com/qna/detail.naver?d1id=11...  
2   https://kin.naver.com/qna/detail.naver?d1id=6&...  
3   https://kin.naver.com/qna/detail.naver?d1id=6&...  
4   h

In [ ]:
# 추출한 데이터 전처리 작업
# 중복 데이터 제거
clean_df = total_df.drop_duplicates(subset=['title'])

# 결측치 처리
clean_df = clean_df.dropna()

# 텍스트 길이 본석후 처리 : AI학습에서 짧은 문장은 정보량이 부족 => 제거
# 텍스트 길이 평균보다 큰 내용들만 추출

# 텍스트 길이의 평균
mean_len = clean_df['title'].str.len().mean()

# 평균보다 큰 행들만 추출
final_df = clean_df.loc[clean_df['title'].str.len() > mean_len]

# 인덱스 리셋
final_df = final_df.reset_index(drop=True)
# 결과 확인, title(질문 제목), content(질문 내용)만 확인
# print(final_df)

# kin_키워드_limit200.csv

                        title  \
0      국민취업지원제도는 어떻게 진행을 하나요?   
1     실업급여 받는 중에 취업하면 어떻게...    
2  실업급여 수급 중 조기 취업 시 불이익은...    
3    소방안전관리자2급 자격증으로 취업가능...    
4   대학교 3학년에 취업하면 자퇴밖에 답이...    

                                                link  
0  https://kin.naver.com/qna/detail.naver?d1id=11...  
1  https://kin.naver.com/qna/detail.naver?d1id=6&...  
2  https://kin.naver.com/qna/detail.naver?d1id=6&...  
3  https://kin.naver.com/qna/detail.naver?d1id=11...  
4  https://kin.naver.com/qna/detail.naver?d1id=4&...  
